# 🐕 Stray Dogs Tunisia — Système Complet
**ESPRIT — 3ème année cycle Ingénieur IA | Axe Déchets & Urbanisme**

## Comment utiliser ce notebook ?
1. **Cellule SETUP** : exécuter une seule fois pour préparer l'environnement
2. **Cellule ENTRAÎNEMENT** : générer les données synthétiques et entraîner les modèles
3. **Cellule INPUT** : remplir les informations de ton quartier
4. **Cellule RUN** : lancer le pipeline → obtenir les 3 solutions
5. **Cellules RÉSULTATS** : visualiser chaque solution en détail

---
| Solution | Description | Approche technique |
|---|---|---|
| S1 | Estimation densité canine | Random Forest + Gradient Boosting (ensemble) |
| S2 | Recommandation & placement des bennes | Formule calibrée + OSM Overpass + scoring géospatial |
| S3 | Identification Feeding Zones | Score AHP multicritère |
| CV | Détection infractions nourrissage | YOLOv8n + Optical Flow + Backtracking vidéo |

## 📦 ÉTAPE 1 — Setup (exécuter une seule fois)

In [1]:
import sys, os
from pathlib import Path

# Ajouter src/ au path
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR      = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image, HTML

from config import ROOT, DATA_SYN, MDL_DIR, VIZ_DIR
from pipeline.pipeline import run_pipeline, load_models
from utils.osm_fetcher import get_district_data, extract_elements_from_png
from utils.visualizer import plot_full_dashboard, plot_interactive_map_html

print('✅ Imports OK')
print(f'   Projet root : {PROJECT_ROOT}')

✅ Imports OK
   Projet root : C:\Users\azizb\Downloads\stray_dogs_v2_complet\integration\stray_dogs_v2


## 🧠 ÉTAPE 2 — Génération données synthétiques + Entraînement

In [2]:
from models.generate_synthetic_data import generate_full_dataset

# Génération du dataset (1200 secteurs synthétiques, 24 gouvernorats)
print('Génération du dataset synthétique...')
df_synthetic = generate_full_dataset()

print(f'\nAperçu du dataset :')
display(df_synthetic[['gouvernorat','zone_type','population','nb_food_poi','nb_bennes_org','nb_chiens','risk_class']].head(10))

Génération du dataset synthétique...
GÉNÉRATION DU DATASET SYNTHÉTIQUE
Couverture : 24 gouvernorats tunisiens (RGPH 2014)
Secteurs   : 24 × 50 = 1200 samples
  ✓ Tunis                : 50 secteurs générés (pop moy: 24,136)
  ✓ Ariana               : 50 secteurs générés (pop moy: 14,532)
  ✓ Ben Arous            : 50 secteurs générés (pop moy: 15,932)
  ✓ Manouba              : 50 secteurs générés (pop moy: 9,067)
  ✓ Nabeul               : 50 secteurs générés (pop moy: 17,261)
  ✓ Zaghouan             : 50 secteurs générés (pop moy: 3,912)
  ✓ Bizerte              : 50 secteurs générés (pop moy: 12,129)
  ✓ Beja                 : 50 secteurs générés (pop moy: 5,907)
  ✓ Jendouba             : 50 secteurs générés (pop moy: 10,460)
  ✓ Kef                  : 50 secteurs générés (pop moy: 6,146)
  ✓ Siliana              : 50 secteurs générés (pop moy: 5,026)
  ✓ Sousse               : 50 secteurs générés (pop moy: 15,173)
  ✓ Monastir             : 50 secteurs générés (pop moy: 11,865)
  

,gouvernorat,zone_type,population,nb_food_poi,nb_bennes_org,nb_chiens,risk_class
0,Tunis,commercial,12115,19,20,1586,2
1,Tunis,residential,13415,5,18,965,1
2,Tunis,commercial,84741,32,124,14365,2
3,Tunis,park,15059,0,23,1349,1
4,Tunis,commercial,31712,32,31,5026,2
5,Tunis,commercial,33189,21,47,5790,2
6,Tunis,residential,22175,4,27,1349,1
7,Tunis,mixed,24028,21,30,2706,2
8,Tunis,industrial,31003,1,43,2632,1
9,Tunis,residential,9729,5,10,1140,2


In [3]:
from models.train_models import run as train_all

# Entraînement de tous les modèles
# Random Forest Regressor + Random Forest Classifier + Gradient Boosting
rf_reg, gb_reg, rf_clf, scaler = train_all(evaluate=True)

# Afficher les résultats d'entraînement
img_path = VIZ_DIR / 'training_results.png'
if img_path.exists():
    fig, ax = plt.subplots(figsize=(20, 12), facecolor='#0d1117')
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print('✅ Modèles entraînés et sauvegardés dans models/trained/')

ENTRAÎNEMENT DES MODÈLES — Stray Dogs Tunisia
[TRAIN] Dataset chargé : 1,200 samples, 9 features
[TRAIN] Train : 960 | Test : 240

[TRAIN] ── Random Forest Regressor ──
  MAE  : 175.0 chiens
  RMSE : 280.7 chiens
  R²   : 0.9319
  CV R²: 0.9005 ± 0.0321

  Feature importance (top 5) :
    population                : 0.8126
    nb_bennes_org             : 0.0506
    nb_food_poi               : 0.0487
    zone_coeff                : 0.0272
    nb_menages                : 0.0262

  ✅ Sauvegardé : models/trained/rf_regressor.joblib

[TRAIN] ── Random Forest Classifier (risque) ──
  Accuracy : 0.8583

  Rapport de classification :
              precision    recall  f1-score   support

      Faible      0.000     0.000     0.000         2
       Moyen      0.873     0.886     0.879       140
       Élevé      0.837     0.837     0.837        98

    accuracy                          0.858       240
   macro avg      0.570     0.574     0.572       240
weighted avg      0.851     0.858     0.

## 🏙️ ÉTAPE 3 — INPUT : Définir le quartier

### Option A — Par nom (OpenStreetMap récupère tout automatiquement)
### Option B — Manuellement (si pas d'accès internet ou quartier inconnu)
### Option C — Depuis un PNG (carte avec éléments colorés)

In [4]:
# ═══════════════════════════════════════════════════════════
#  ▶ REMPLIR ICI — Informations sur le quartier
# ═══════════════════════════════════════════════════════════

# ── Informations de base ────────────────────────────────────
DISTRICT_NAME  = 'La Marsa'          # Nom du quartier/ville
COUNTRY        = 'Tunisia'           # Pays

# ── Si tu connais la bbox (optionnel, sinon OSM la trouve) ──
MANUAL_BBOX = None  # ou dict ex: {'lat_min': 36.862, 'lat_max': 36.930, 'lon_min': 10.295, 'lon_max': 10.350}

# ── Informations complémentaires ────────────────────────────
POPULATION    = 92987     # Nombre d'habitants (RGPH 2014)
NB_MENAGES    = 26062     # Nombre de ménages
SURFACE_KM2   = 12.5      # Surface en km²
ZONE_TYPE     = 'mixed'   # residential | commercial | mixed | beach | periphery | ...
N_ZONES       = 7         # Nombre de feeding zones à identifier

# ── PNG de carte (optionnel) ────────────────────────────────
PNG_PATH = None  # ex: '../data/raw/carte_lamarsa.png'
#
# Convention couleurs sur le PNG :
#   Rouge  = bennes organiques
#   Bleu   = bennes plastique
#   Orange = restaurants/cafés
#   Vert   = espaces verts/parcs
#   Jaune  = écoles

print(f'✅ Configuration : {DISTRICT_NAME}')
print(f'   Population  : {POPULATION:,}')
print(f'   Ménages     : {NB_MENAGES:,}')
print(f'   Surface     : {SURFACE_KM2} km²')
print(f'   Zone        : {ZONE_TYPE}')
print(f'   PNG         : {PNG_PATH or "Non fourni → OSM"}')

✅ Configuration : La Marsa
   Population  : 92,987
   Ménages     : 26,062
   Surface     : 12.5 km²
   Zone        : mixed
   PNG         : Non fourni → OSM


## 🗺️ ÉTAPE 4 — Récupération des données OSM

In [5]:
# Récupération des données du quartier
print(f'Récupération des données pour : {DISTRICT_NAME}...')

district_data = get_district_data(
    district_name=DISTRICT_NAME,
    bbox=MANUAL_BBOX,
    country=COUNTRY,
)

BBOX      = district_data['bbox']
food_poi  = district_data['food_poi']
schools   = district_data['schools']
parks     = district_data['parks']

print(f'\n✅ Données récupérées :')
print(f'   Bbox       : lat [{BBOX["lat_min"]:.3f}, {BBOX["lat_max"]:.3f}]')
print(f'   Food POI   : {len(food_poi)}')
print(f'   Écoles     : {len(schools)}')
print(f'   Parcs      : {len(parks)}')

Récupération des données pour : La Marsa...
[OSM] Recherche : 'La Marsa, Tunisia'
[OSM] Zone : المرسى, تونس, ولاية تونس, 2070, تونس
[OSM] Bbox : lat [36.839, 36.919] lon [10.288, 10.368]
[OSM] POST → https://overpass-api.de/api/interpreter
[OSM] ✓ 1635 POI — 410 food · 57 écoles · 1156 parcs

✅ Données récupérées :
   Bbox       : lat [36.839, 36.919]
   Food POI   : 410
   Écoles     : 57
   Parcs      : 1156


In [6]:
# ── Bennes : depuis PNG ou saisie manuelle ───────────────────

# Option A : extraction depuis PNG
bins_from_png = {}
if PNG_PATH and Path(PNG_PATH).exists():
    bins_from_png = extract_elements_from_png(PNG_PATH, BBOX)
    print(f'Éléments extraits du PNG : {sum(len(v) for v in bins_from_png.values())} points')
    
    # Construire le DataFrame des bennes depuis le PNG
    bins_rows = []
    for lat, lon in bins_from_png.get('bins_org', []):
        bins_rows.append({'lat': lat, 'lon': lon, 'type': 'organique'})
    for lat, lon in bins_from_png.get('bins_plas', []):
        bins_rows.append({'lat': lat, 'lon': lon, 'type': 'plastique'})
    BINS_DF = pd.DataFrame(bins_rows)
    
    # Enrichir food_poi et schools depuis le PNG
    if bins_from_png.get('food_poi'):
        extra_food = pd.DataFrame(bins_from_png['food_poi'], columns=['lat','lon'])
        extra_food['type'] = 'restaurant'; extra_food['category'] = 'food'; extra_food['source'] = 'png'
        food_poi = pd.concat([food_poi, extra_food], ignore_index=True)
    if bins_from_png.get('schools'):
        extra_schools = pd.DataFrame(bins_from_png['schools'], columns=['lat','lon'])
        extra_schools['category'] = 'school'; extra_schools['source'] = 'png'
        schools = pd.concat([schools, extra_schools], ignore_index=True)

else:
    # Option B : bennes simulées (distribution réaliste)
    print('Pas de PNG → génération de bennes simulées...')
    from src.utils.bin_generator import generate_bins_for_bbox
    BINS_DF = generate_bins_for_bbox(BBOX, POPULATION)
    print(f'   {len(BINS_DF)} bennes générées')

print(f'\n✅ Bennes : {len(BINS_DF)} total')
if not BINS_DF.empty:
    print(BINS_DF['type'].value_counts().to_string())

Pas de PNG → génération de bennes simulées...
[BIN_GEN] 224 bennes générées (132 organiques, 92 plastique)
   224 bennes générées

✅ Bennes : 224 total
type
organique    132
plastique     92


## 🚀 ÉTAPE 5 — Lancement du pipeline (3 solutions)

In [7]:
# Chargement des modèles entraînés
models = load_models()

# Pipeline complet → 3 solutions
results = run_pipeline(
    district_name = DISTRICT_NAME,
    population    = POPULATION,
    nb_menages    = NB_MENAGES,
    surface_km2   = SURFACE_KM2,
    bbox          = BBOX,
    bins_df       = BINS_DF,
    food_poi_df   = food_poi,
    schools_df    = schools,
    parks_df      = parks,
    zone_type     = ZONE_TYPE,
    n_zones       = N_ZONES,
    models        = models,
    verbose       = True,
)

s1 = results['s1']
s2 = results['s2']
s3 = results['s3']

print('\n✅ Pipeline terminé.')

[PIPELINE] Modèles chargés ✓

PIPELINE — La Marsa

[S1] Estimation densité canine…
  → 11467 chiens estimés | Risque : Élevé (87%)

[S2] Optimisation des bennes…
[BinPlacer] Nombre recommandé : 300
[BinPlacer] 92987 hab ÷ 150 = 619.9 + 192 restos×0.25 = 48.0 + 134 cafés×0.12 = 16.1 → total brut 1257.3 → 300 bennes
[BinPlacer] Overpass indisponible (406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter) — fallback activé
[BinPlacer] Fallback : 1228 candidats générés
[BinPlacer] 183 bennes sélectionnées (demandées : 300)
  → 300 bennes recommandées | 183 placées | 132 points noirs

[S3] Identification feeding zones…
  → 7 feeding zones | Meilleur score : 99.9%

✅ Pipeline terminé.


## 📊 ÉTAPE 6 — Résultats

In [8]:
# ─── SOLUTION 1 : Densité canine ───────────────────────────
print('═'*55)
print('  SOLUTION 1 — ESTIMATION DENSITÉ CANINE')
print('═'*55)
print(f'''
  Quartier       : {DISTRICT_NAME}
  Population     : {POPULATION:,} habitants
  ──────────────────────────────────────────
  Chiens estimés : {s1["nb_chiens"]:,}
  Ratio          : {s1["details"]["ratio_chien_habitant"]}
  Niveau risque  : {s1["risk_label"]}
  Confiance      : {s1["confidence"]*100:.0f}%
  Modèle         : {s1["source"]}
  ──────────────────────────────────────────
  Contribution POI alimentaires : +{s1["details"]["contribution_food_poi"]} chiens
  Contribution bennes org       : +{s1["details"]["contribution_bennes"]} chiens
  Coeff zone ({ZONE_TYPE})     : ×{s1["details"]["zone_coeff"]}
''')

═══════════════════════════════════════════════════════
  SOLUTION 1 — ESTIMATION DENSITÉ CANINE
═══════════════════════════════════════════════════════

  Quartier       : La Marsa
  Population     : 92,987 habitants
  ──────────────────────────────────────────
  Chiens estimés : 11,467
  Ratio          : 1 chien / 8 habitants
  Niveau risque  : Élevé
  Confiance      : 87%
  Modèle         : RF+GB (ensemble)
  ──────────────────────────────────────────
  Contribution POI alimentaires : +820 chiens
  Contribution bennes org       : +396 chiens
  Coeff zone (mixed)     : ×1.3



In [9]:
# ─── SOLUTION 2 : Bennes recommandées ───────────────────────
print('═'*60)
print('  SOLUTION 2 — RECOMMANDATION & PLACEMENT DES BENNES')
print('═'*60)

st  = s2['stats']
bci = s2.get('bin_count_info', {})
rec = s2.get('recommended_bins', None)

print(f'''
  ── Formule de recommandation ─────────────────────────────
  {st.get("formule", "N/A")}

  ── Résumé ───────────────────────────────────────────────
  Bennes recommandées  : {st.get("nb_recommande", "—")}
  Bennes effectivement placées : {st.get("bennes_placees", "—")}
  Intersections OSM trouvées   : {st.get("intersections_osm", "—")}
  Restaurants détectés         : {st.get("nb_restaurants", "—")}
  Cafés détectés               : {st.get("nb_cafes", "—")}
  Points noirs (bennes critiques) : {st.get("black_spots", "—")}

  ── Décomposition des contributions ──────────────────────''')

contrib = st.get("contributions", {})
for k, v in contrib.items():
    print(f'  {k:<20s} : +{v} bennes')

print()
if rec is not None and not rec.empty:
    print(f'\n  Bennes recommandées (top 10) :')
    display(rec[['lat','lon','score','type','source']].head(10).round(5))
else:
    print('  (aucune benne placée — vérifier la bbox ou les paramètres)')

════════════════════════════════════════════════════════════
  SOLUTION 2 — RECOMMANDATION & PLACEMENT DES BENNES
════════════════════════════════════════════════════════════

  ── Formule de recommandation ─────────────────────────────
  92987 hab ÷ 150 = 619.9 + 192 restos×0.25 = 48.0 + 134 cafés×0.12 = 16.1 → total brut 1257.3 → 300 bennes

  ── Résumé ───────────────────────────────────────────────
  Bennes recommandées  : 300
  Bennes effectivement placées : 183
  Intersections OSM trouvées   : 1228
  Restaurants détectés         : 192
  Cafés détectés               : 134
  Points noirs (bennes critiques) : 132

  ── Décomposition des contributions ──────────────────────
  base_population      : +619.9 bennes
  restaurants          : +48.0 bennes
  cafes                : +16.1 bennes
  bonus_chiens         : +573.4 bennes


  Bennes recommandées (top 10) :


,lat,lon,score,type,source
0,36.87895,10.32797,1578.816,organique,near_food_poi
1,36.87964,10.32652,1060.210,organique,near_food_poi
2,36.88042,10.32823,841.753,organique,near_food_poi
3,36.87741,10.32757,559.447,organique,near_food_poi
4,36.87987,10.32485,204.031,organique,near_food_poi
5,36.88374,10.33202,60.299,organique,near_food_poi
6,36.87268,10.33979,46.348,organique,near_food_poi
7,36.88559,10.32344,45.186,organique,near_food_poi
8,36.88255,10.33292,42.168,organique,near_food_poi
9,36.86969,10.34357,40.148,organique,near_food_poi


In [10]:
# ─── SOLUTION 3 : Feeding Zones ───────────────────────────
print('═'*55)
print('  SOLUTION 3 — FEEDING ZONES')
print('═'*55)

fz = s3['feeding_zones']
if not fz.empty:
    print(f'  {len(fz)} zones identifiées\n')
    display(fz[['zone_id','lat','lon','score_pct','rayon_m']].round(4))
else:
    print('  Aucune zone (grille vide — vérifier la bbox)')

═══════════════════════════════════════════════════════
  SOLUTION 3 — FEEDING ZONES
═══════════════════════════════════════════════════════
  7 zones identifiées



,zone_id,lat,lon,score_pct,rayon_m
0,FZ-01,36.8941,10.3117,99.9,400
1,FZ-02,36.8661,10.3187,99.9,400
2,FZ-03,36.8681,10.3277,99.8,400
3,FZ-04,36.8921,10.3207,99.8,400
4,FZ-05,36.8831,10.3157,99.8,400
5,FZ-06,36.9101,10.3017,99.7,400
6,FZ-07,36.8751,10.3487,99.4,400


In [ ]:
# ─── SOLUTION 3 : Génération détaillée des Feeding Zones ──────
# Montre explicitement comment les zones sont calculées :
#  1. Grille de candidats (~100m × ~100m)
#  2. Score AHP multicritère par cellule
#  3. Sélection avec contrainte de diversité spatiale (~900m)

from pipeline.pipeline import solution3_feeding_zones
from config import GRID_STEP_DEG, MIN_DIST_FZ_DEG, AHP_WEIGHTS, DBSCAN_EPS, DBSCAN_MIN_SAMPLES

# ─── 1. Paramètres du modèle ────────────────────────────────────
print('═'*60)
print('  GÉNÉRATION DES FEEDING ZONES — Détail complet')
print('═'*60)
print(f'''
  ── Grille spatiale ─────────────────────────────────────────
  Résolution     : GRID_STEP_DEG = {GRID_STEP_DEG} deg (~100 m)
  Bbox La Marsa  : Δlat={BBOX["lat_max"]-BBOX["lat_min"]:.3f}° × Δlon={BBOX["lon_max"]-BBOX["lon_min"]:.3f}°
  Cellules total : ~{int((BBOX["lat_max"]-BBOX["lat_min"])/GRID_STEP_DEG) * int((BBOX["lon_max"]-BBOX["lon_min"])/GRID_STEP_DEG):,} points

  ── Score AHP multicritère (6 critères) ─────────────────────
  Critère                  Poids   Logique
  ─────────────────────────────────────────────────────────────
  dog_density              {AHP_WEIGHTS["dog_density"]:.2f}    densité canine élevée → score ↑
  dist_schools             {AHP_WEIGHTS["dist_schools"]:.2f}    loin des écoles       → score ↑
  near_green               {AHP_WEIGHTS["near_green"]:.2f}    proche parc/espace vert → score ↑
  dist_food_poi            {AHP_WEIGHTS["dist_food_poi"]:.2f}    loin des restos/cafés  → score ↑
  dist_org_bins            {AHP_WEIGHTS["dist_org_bins"]:.2f}    loin des bennes org.   → score ↑
  road_access              {AHP_WEIGHTS["road_access"]:.2f}    dans la bbox (accès)   → score ↑
  ─────────────────────────────────────────────────────────────
  Total poids              {sum(AHP_WEIGHTS.values()):.2f}

  ── Contrainte diversité spatiale ───────────────────────────
  Distance minimale entre zones : MIN_DIST_FZ_DEG = {MIN_DIST_FZ_DEG} deg (~{MIN_DIST_FZ_DEG*111:.0f} m)
  Zones cibles                  : N_ZONES = {N_ZONES}
''')

# ─── 2. Ré-appel direct de solution3_feeding_zones ─────────────
print('  Calcul en cours...')
lat_c = (BBOX["lat_min"] + BBOX["lat_max"]) / 2
lon_c = (BBOX["lon_min"] + BBOX["lon_max"]) / 2
import numpy as np, pandas as pd
dog_map_local = pd.DataFrame([{"lat": lat_c, "lon": lon_c, "nb_chiens": s1["nb_chiens"]}])

fz_result = solution3_feeding_zones(
    bbox            = BBOX,
    dog_density_map = dog_map_local,
    food_poi_df     = food_poi,
    schools_df      = schools,
    parks_df        = parks,
    org_bins_df     = BINS_DF,
    n_zones         = N_ZONES,
)

df_fz_detail, df_grid_full = fz_result
print(f'  ✅ Grille : {len(df_grid_full):,} cellules évaluées')
print(f'  ✅ Zones sélectionnées : {len(df_fz_detail)}\n')

# ─── 3. Top 20 candidats (avant sélection diversité) ───────────
print('─'*60)
print(f'  TOP 20 candidats (classés par score AHP brut)')
print('─'*60)
top20 = df_grid_full.nlargest(20, 'score')[['lat', 'lon', 'score']].copy()
top20['score_pct'] = (top20['score'] * 100).round(1)
top20['rank'] = range(1, 21)
top20 = top20[['rank', 'lat', 'lon', 'score_pct']].reset_index(drop=True)
display(top20)

# ─── 4. Distribution des scores sur la grille ──────────────────
print(f'\n  Statistiques du score AHP sur toute la grille :')
desc = df_grid_full['score'].describe()
print(f'  Min    : {desc["min"]*100:.1f}%')
print(f'  Médiane: {desc["50%"]*100:.1f}%')
print(f'  Moyen  : {desc["mean"]*100:.1f}%')
print(f'  Max    : {desc["max"]*100:.1f}%')
print(f'  Cellules score > 90% : {(df_grid_full["score"] > 0.90).sum()}')
print(f'  Cellules score > 95% : {(df_grid_full["score"] > 0.95).sum()}')

# ─── 5. Zones finales avec scores détaillés ────────────────────
print(f'\n{"─"*60}')
print(f'  ZONES FINALES — après contrainte ~{MIN_DIST_FZ_DEG*111:.0f}m entre zones')
print(f'{"─"*60}')
display(df_fz_detail[['zone_id', 'lat', 'lon', 'score_pct', 'rayon_m']].round(4))

# ─── 6. Visualisation rapide des scores (barre horizontale) ────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0d1117')
fig.suptitle(f'Feeding Zones — Score AHP | {DISTRICT_NAME}', color='white', fontsize=14, fontweight='bold')

# Panel gauche : barres de scores par zone
ax1 = axes[0]
ax1.set_facecolor('#161b22')
colors = ['#2ecc71' if s >= 99 else '#f39c12' if s >= 95 else '#e74c3c'
          for s in df_fz_detail['score_pct']]
bars = ax1.barh(df_fz_detail['zone_id'][::-1], df_fz_detail['score_pct'][::-1],
               color=colors[::-1], edgecolor='#30363d', height=0.6)
for bar, val in zip(bars, df_fz_detail['score_pct'][::-1]):
    ax1.text(bar.get_width() - 1.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', ha='right', color='white', fontsize=10, fontweight='bold')
ax1.set_xlim(85, 101)
ax1.set_xlabel('Score AHP (%)', color='white')
ax1.set_title('Score par zone', color='white', fontsize=11)
ax1.tick_params(colors='white')
ax1.spines['bottom'].set_color('#30363d')
ax1.spines['left'].set_color('#30363d')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_facecolor('#161b22')

# Panel droit : poids AHP (pie chart)
ax2 = axes[1]
ax2.set_facecolor('#161b22')
labels_ahp  = ['Densité canine\n(30%)', 'Dist. écoles\n(20%)', 'Espaces verts\n(15%)',
               'Dist. food POI\n(15%)', 'Dist. bennes\n(10%)', 'Accès route\n(10%)']
sizes_ahp   = list(AHP_WEIGHTS.values())
palette_ahp = ['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6','#1abc9c']
wedges, texts, autotexts = ax2.pie(sizes_ahp, labels=labels_ahp, colors=palette_ahp,
                                    autopct='%1.0f%%', startangle=90,
                                    textprops={'color': 'white', 'fontsize': 9})
for at in autotexts:
    at.set_color('white')
    at.set_fontsize(9)
ax2.set_title('Poids AHP', color='white', fontsize=11)

plt.tight_layout()
plt.savefig(str(VIZ_DIR / 'feeding_zones_ahp_detail.png'), dpi=120,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'✅ Graphique sauvegardé : visualizations/feeding_zones_ahp_detail.png')

In [11]:
# ─── Dashboard visuel ─────────────────────────────────────
print('Génération du dashboard...')

out_path = plot_full_dashboard(
    district_name = DISTRICT_NAME,
    bbox          = BBOX,
    s1_result     = s1,
    s2_result     = s2,
    s3_result     = s3,
    bins_df       = BINS_DF,
    food_poi_df   = food_poi,
    schools_df    = schools,
    parks_df      = parks,
    save          = True,
)

# Afficher dans le notebook
fig, ax = plt.subplots(figsize=(22, 15), facecolor='#0d1117')
ax.imshow(mpimg.imread(out_path))
ax.axis('off')
plt.tight_layout()
plt.show()
print(f'✅ Dashboard : {out_path}')

Génération du dashboard...
[VIZ] Dashboard sauvegardé : C:\Users\azizb\Downloads\stray_dogs_v2_complet\integration\stray_dogs_v2\visualizations\dashboard_la_marsa.png
✅ Dashboard : C:\Users\azizb\Downloads\stray_dogs_v2_complet\integration\stray_dogs_v2\visualizations\dashboard_la_marsa.png


In [12]:
# ─── Carte interactive HTML (si folium installé) ──────────
html_path = plot_interactive_map_html(
    district_name = DISTRICT_NAME,
    bbox          = BBOX,
    s1            = s1,
    s2            = s2,
    s3            = s3,
    bins_df       = BINS_DF,
    food_poi_df   = food_poi,
    schools_df    = schools,
    parks_df      = parks,
)

if html_path:
    print(f'✅ Ouvrir dans le navigateur : {html_path}')
    display(HTML(f'<a href="{html_path}" target="_blank">🗺️ Ouvrir la carte interactive</a>'))

[VIZ] Carte interactive sauvegardée : C:\Users\azizb\Downloads\stray_dogs_v2_complet\integration\stray_dogs_v2\visualizations\map_la_marsa.html
✅ Ouvrir dans le navigateur : C:\Users\azizb\Downloads\stray_dogs_v2_complet\integration\stray_dogs_v2\visualizations\map_la_marsa.html


In [13]:
# ─── Export des résultats en CSV ──────────────────────────
from config import DATA_OUT
slug = DISTRICT_NAME.lower().replace(' ', '_')

rec_bins = s2.get('recommended_bins', None)
if rec_bins is not None and not rec_bins.empty:
    rec_bins.to_csv(DATA_OUT / f'{slug}_bennes_recommandees.csv', index=False)
    print(f'✅ Bennes exportées : {len(rec_bins)} lignes')

if not s3['feeding_zones'].empty:
    s3['feeding_zones'].to_csv(DATA_OUT / f'{slug}_feeding_zones.csv', index=False)

import json
st = s2.get('stats', {})
summary = {
    'district':            DISTRICT_NAME,
    'population':          POPULATION,
    'chiens_estimes':      s1['nb_chiens'],
    'niveau_risque':       s1['risk_label'],
    'confidence':          s1['confidence'],
    'bennes_recommandees': st.get('nb_recommande', 0),
    'bennes_placees':      st.get('bennes_placees', 0),
    'intersections_osm':   st.get('intersections_osm', 0),
    'black_spots':         st.get('black_spots', 0),
    'feeding_zones':       len(s3['feeding_zones']),
}
with open(DATA_OUT / f'{slug}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('✅ Résultats exportés dans data/outputs/')
print(json.dumps(summary, indent=2, ensure_ascii=False))

✅ Bennes exportées : 183 lignes
✅ Résultats exportés dans data/outputs/
{
  "district": "La Marsa",
  "population": 92987,
  "chiens_estimes": 11467,
  "niveau_risque": "Élevé",
  "confidence": 0.874,
  "bennes_recommandees": 300,
  "bennes_placees": 183,
  "intersections_osm": 1228,
  "black_spots": 132,
  "feeding_zones": 7
}
